# 🫧 K-Means Clustering
**ISLP Chapter 12 · Data Pattern Recognition for the Rest of Us**

> K-means groups customers into segments by finding cluster centers that minimize within-cluster distance. The result: actionable customer segments you can market to differently.

### Dataset
**UCI Online Retail — Customer RFM Analysis**
Real e-commerce transaction data from a UK gift retailer (Chen et al. 2012, UCI ML Repository).
4,372 customers. RFM features derived from actual purchase behaviour:
- **Recency** — days since last purchase
- **Frequency** — number of distinct orders
- **Monetary** — total spend (£)

### Time: ~55 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess, os, warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
print("\u2713 Setup complete")

In [ ]:
# Load UCI Online Retail RFM dataset from course repo
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)
DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '../data'

rfm = pd.read_csv(f'{DATA_DIR}/OnlineRetailRFM.csv')

print("UCI Online Retail — Customer RFM Dataset")
print(f"Source: Chen et al. 2012, UCI ML Repository")
print(f"4,372 unique customers, UK gift retailer, Dec 2010 - Dec 2011")
print(f"\nRFM distributions:")
print(rfm[['Recency','Frequency','Monetary']].describe().round(1).to_string())
print(f"\nInterpretation:")
print(f"  Recency:   days since last purchase  (lower = more recent = better)")
print(f"  Frequency: number of distinct orders  (higher = better)")
print(f"  Monetary:  total spend in GBP  (higher = better)")
print(f"\n  Recency median {rfm.Recency.median():.0f}d — half of customers haven't bought in 3+ months")
print(f"  Frequency median {rfm.Frequency.median():.0f} orders — most customers buy only once or twice")
print(f"  Monetary median \xa3{rfm.Monetary.median():,.0f} — strong right skew, a few high-value customers")

## 📐 Part 1 — Why RFM? The Business Logic

RFM is the gold standard framework for customer analytics because it captures three dimensions that predict future behaviour:

| Dimension | What it measures | Why it matters |
|-----------|-----------------|----------------|
| **Recency** | How recently did they buy? | Recent buyers are more likely to buy again |
| **Frequency** | How often do they buy? | Frequent buyers have stronger loyalty |
| **Monetary** | How much do they spend? | High spenders generate disproportionate revenue |

**The Pareto principle in retail:** typically 20% of customers generate 80% of revenue. RFM clustering finds that 20%.

**K-means** assigns each customer to the nearest cluster centroid and updates centroids iteratively. We must standardize first — £Monetary would completely dominate days-Recency otherwise.

In [ ]:
# Log-transform Monetary (heavy right skew) then standardize all features
rfm_model = rfm.copy()
rfm_model['Monetary_log'] = np.log1p(rfm_model['Monetary'])
features = ['Recency', 'Frequency', 'Monetary_log']

X = rfm_model[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Invert Recency so higher score always = better (for intuitive segment naming)
X_scaled[:, 0] *= -1

# Show why standardization matters
print("=== Why we must standardize ===\n")
print(f"{'Feature':<15} {'Raw range':<20} {'Std range':<20}")
print("-"*55)
for i, feat in enumerate(features):
    raw  = X[:, i]
    std  = X_scaled[:, i]
    print(f"{feat:<15} [{raw.min():>6.0f}, {raw.max():>6.0f}]    [{std.min():>5.2f}, {std.max():>5.2f}]")

print("\n  Without scaling: Monetary (0-80,000) dominates Recency (1-365)")
print("  With scaling: all three features contribute equally to distance")

In [ ]:
# Elbow + Silhouette to choose K
wcss = []
sil  = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, labels, sample_size=1000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), wcss, 'o-', color='#1e5fa8', lw=2, markersize=7)
axes[0].set_xlabel('Number of Segments (K)')
axes[0].set_ylabel('WCSS')
axes[0].set_title('Elbow Method — look for the bend')

best_k = list(K_range)[sil.index(max(sil))]
axes[1].plot(list(K_range), sil, 's-', color='#e85d20', lw=2, markersize=7)
axes[1].axvline(best_k, color='#1a7a45', lw=1.5, ls='--',
                label=f'Best K={best_k}  (silhouette={max(sil):.3f})')
axes[1].set_xlabel('Number of Segments (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — higher = more distinct segments')
axes[1].legend()

plt.suptitle('Choosing K — UCI Online Retail Customers', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
print(f"\n\u2714 Silhouette suggests K={best_k} distinct customer segments")

In [ ]:
# Fit final model
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm['Cluster'] = km_final.fit_predict(X_scaled)

# Profile each cluster on ORIGINAL (untransformed) scale
profile = rfm.groupby('Cluster').agg(
    N=('CustomerID','count'),
    Avg_Recency=('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary=('Monetary','mean'),
    Med_Monetary=('Monetary','median'),
).round(1)
profile['Pct'] = (profile['N'] / len(rfm) * 100).round(1)

# Name segments based on profile
def name_seg(row):
    if row['Avg_Recency'] < 40 and row['Avg_Frequency'] >= 6:
        return 'Champions'
    elif row['Avg_Recency'] < 70 and row['Avg_Frequency'] >= 4:
        return 'Loyal Customers'
    elif row['Avg_Recency'] > 200:
        return 'Lost / Inactive'
    elif row['Avg_Frequency'] <= 1.5:
        return 'New / One-Time'
    else:
        return 'At-Risk'

profile['Segment'] = profile.apply(name_seg, axis=1)
rfm['Segment'] = rfm['Cluster'].map(profile['Segment'].to_dict())

print("=== Customer Segment Profiles (UCI Online Retail) ===\n")
display_cols = ['Segment','N','Pct','Avg_Recency','Avg_Frequency','Avg_Monetary','Med_Monetary']
print(profile[display_cols].sort_values('Avg_Monetary', ascending=False).to_string())

print("\n\u2714 Revenue concentration:")
for _, row in profile.sort_values('Avg_Monetary', ascending=False).iterrows():
    rev_share = row['N'] * row['Avg_Monetary'] / (rfm['Monetary'].sum()) * 100
    print(f"  {row['Segment']:<20} {row['Pct']:>5.1f}% of customers  "
          f"\xa3{row['Avg_Monetary']:>8,.0f} avg spend  "
          f"{rev_share:>5.1f}% of total revenue")

In [ ]:
# Visualise segments in PCA space + profile bar chart
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

seg_order = profile.sort_values('Avg_Monetary', ascending=False)['Segment'].tolist()
colors = ['#1a7a45','#1e5fa8','#e85d20','#6b46c1','#0e7490']
seg_color = {s: colors[i] for i, s in enumerate(seg_order)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA scatter
for seg in seg_order:
    mask = rfm['Segment'] == seg
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    s=8, alpha=0.5, color=seg_color[seg], label=seg)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% variance)')
axes[0].set_title('Customer Segments in PCA Space\n(UCI Online Retail)')
axes[0].legend(fontsize=9, markerscale=3)

# Normalised profile
profile_norm = profile[['Avg_Recency','Avg_Frequency','Avg_Monetary']].copy()
# For Recency: invert so bar = higher is better
profile_norm['Avg_Recency'] = profile_norm['Avg_Recency'].max() - profile_norm['Avg_Recency']
for col in profile_norm.columns:
    mn, mx = profile_norm[col].min(), profile_norm[col].max()
    profile_norm[col] = (profile_norm[col]-mn)/(mx-mn+1e-9)
profile_norm.index = profile['Segment']
profile_norm.columns = ['Recency (inverted)','Frequency','Monetary']
profile_norm.loc[seg_order].plot(kind='bar', ax=axes[1],
    color=['#1e5fa8','#e85d20','#1a7a45'], edgecolor='white', width=0.7)
axes[1].set_title('Normalised Segment Profiles\n(higher = better on all axes)')
axes[1].set_xlabel('')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=25, ha='right')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()
print("\n\u2714 Business actions by segment:")
actions = {
    'Champions':       'Reward and ask for referrals — they are your best advocates',
    'Loyal Customers': 'Upsell and cross-sell — they are engaged and trust you',
    'At-Risk':         'Win-back campaign — personalised discount before they leave',
    'New / One-Time':  'Onboarding nurture — encourage a second purchase within 30 days',
    'Lost / Inactive': 'Low-cost reactivation email — or accept churn and reduce spend',
}
for seg, action in actions.items():
    if seg in rfm['Segment'].values:
        n_seg = (rfm['Segment']==seg).sum()
        print(f"  {seg:<22} ({n_seg:,} customers) — {action}")

In [ ]:
# @title 📝 Quiz — K-Means Clustering
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does K-means minimise, and why must features be standardised first?
# @markdown **Q2:** What does the silhouette score measure? What does a value near 0 mean?
# @markdown **Q3:** What is the elbow method and why doesn't it always give a clear answer?
# @markdown **Q4:** Why is K-means sensitive to initialisation, and how does n_init help?
# @markdown **Q5:** For the Champions segment: describe ONE concrete marketing action and why it fits.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_="K-Means Clustering"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*